# How much of this tool Prettier already does

This tool removes manual line breaks from Markdown prose. So does
`prettier --prose-wrap never`: a general Markdown formatter with a setting
that joins every soft-wrapped paragraph it can find. Two programs, one
transform. Anyone already running Prettier is entitled to ask what the second
one is for.

They overlap a great deal, and this page is not an argument that they do not.
On the conformance corpus that specifies this tool, the two reach the same
answer for most documents, and where they differ it is nearly always the same
difference in the same direction: Prettier joins lines this tool leaves alone.
The rest of this notebook is about measuring that precisely enough to be
useful to someone deciding between them.

Three things are measured separately, because they are three different
questions with three different answers. How the two programs break prose. What
each does with a comment asking for a paragraph to be left alone. And what each
does to a file whose line endings are not this machine's.

Every number and every chart below is computed by the code cell above it, so
nothing in this text can contradict the output next to it. A number written
into a sentence would be wrong the next time the notebook runs.

## Method

The comparison is between two answers to the same document: the one this tool
produced, and the one `prettier --prose-wrap never` produces. This tool's
answer is a corpus case's `expected.md`, or its `input.md` where the case
declares that nothing should change. That substitution is checked rather than
assumed: a cell below re-runs the tool over a copy of the corpus and refuses to
go on unless it still produces every one of those files byte for byte.

**The two answers cannot be compared as bytes.** Prettier does not only join
prose. It renumbers ordered lists, rewrites tilde fences as backticks, pads
table cells to an even width, converts asterisk emphasis to underscores, and
rewrites CRLF line endings to LF. None of that is a decision about where a line
break belongs, and all of it would count against the comparison. So both
answers are put through Prettier once more under `proseWrap: preserve`, a mode
that applies every one of those cosmetics and moves no line break at all. What
survives that pass is line structure, which is the subject.

**The normalization is asymmetric, and the asymmetry is the whole design.**
Prettier's answer is normalized as `preserve` applied to `never` applied to the
input. This tool's answer is normalized as `preserve` applied to the tool's
output, and nothing else. The tool's side never sees `never`.

Running `never` on both sides is the obvious simplification, and it is wrong in
the way that is hardest to catch: it succeeds. It re-applies Prettier's joining
to this tool's output, so every paragraph the tool declined to join is joined
anyway, and the measurement stops asking whether the two programs agree and
starts asking whether Prettier agrees with itself. The cell below computes that
variant alongside the real one and prints both, because the difference between
them is tens of points rather than a rounding error, and one character of
editing separates the two at all times.

**`proseWrap: preserve` is checked, not trusted.** It is not universally
non-joining: it will pull a link reference definition split over two lines back
onto one. No corpus case contains one today, which is luck rather than design,
so the cell below counts non-blank lines before and after every normalizing
pass and stops the run if a pass ever removed one.

**Every Prettier option is pinned, and the version comes from one clock.**
The parser, the print width, the tab width, the tab setting, the line ending
and the embedded-language setting are all fixed at Prettier's own defaults,
because the claim being tested is about what a reader gets from
`prettier --prose-wrap never` with no configuration of their own; `proseWrap`
is the only thing that varies. The version is read out of the `rev:` this
repository already pins for its Prettier hook, and the run fails if npm
resolves anything else. The registry's idea of the newest release is
deliberately never consulted: a second clock would disagree with the first one
silently, and the page would then describe a Prettier this repository does not
run.

In [1]:
import json
import re
import shutil
import subprocess
from pathlib import Path

# Everything this notebook writes outside the docs directory goes here.
SCRATCH = Path('/tmp/markdown-prose-parity')


def git(*argv: str) -> str:
    """Return the stripped stdout of a git command, or ``''`` if it failed."""
    return subprocess.run(
        ['git', *argv], capture_output=True, text=True, check=False
    ).stdout.strip()


def run(argv: list[str], failure: str) -> str:
    """Return the stdout of ``argv``, ending the run with ``failure`` if it fails."""
    done = subprocess.run(argv, capture_output=True, text=True, check=False)
    if done.returncode != 0:
        raise SystemExit(f'{failure} (exit {done.returncode}){chr(10)}{done.stderr}')
    return done.stdout


REPO = Path(git('rev-parse', '--show-toplevel'))
CORPUS = REPO / 'corpus' / 'cases'
CASE_DIRS = sorted(path for path in CORPUS.iterdir() if path.is_dir())
TOOL = REPO / '.venv/bin/unwrap-markdown-prose-py'
if not TOOL.exists():
    raise SystemExit(f'missing {TOOL}: run `uv sync`')

# The version is the pin on this repository's own Prettier hook, which
# pre-commit.ci moves when Prettier releases. The newest release on the
# registry could be a Prettier nothing in this tree runs.
pinned = re.search(
    r'rbubley/mirrors-prettier\s*\n\s*rev:\s*v(\S+)',
    (REPO / '.pre-commit-config.yaml').read_text(encoding='utf-8'),
)
if pinned is None:
    raise SystemExit(f'no mirrors-prettier rev pinned in {REPO}')
PRETTIER_PIN = pinned.group(1)

NODE_PREFIX = SCRATCH / 'node'
NODE_PREFIX.mkdir(parents=True, exist_ok=True)
run(
    [
        'npm',
        'install',
        '--prefix',
        str(NODE_PREFIX),
        '--no-audit',
        '--no-fund',
        '--loglevel',
        'error',
        f'prettier@{PRETTIER_PIN}',
    ],
    'npm could not install the pinned prettier',
)

PRETTIER_MODULE = NODE_PREFIX / 'node_modules/prettier/index.mjs'
PRETTIER_VERSION = run(
    [
        'node',
        '--input-type=module',
        '--eval',
        f'import p from {json.dumps(str(PRETTIER_MODULE))};'
        'process.stdout.write(p.version)',
    ],
    'node could not load the installed prettier',
).strip()
if PRETTIER_VERSION != PRETTIER_PIN:
    raise SystemExit(
        f'the tree pins prettier {PRETTIER_PIN} and npm resolved {PRETTIER_VERSION}'
    )

NODE_VERSION = run(['node', '--version'], 'node is not runnable').strip()
print(f'prettier  {PRETTIER_VERSION}, the version .pre-commit-config.yaml pins')
print(f'node      {NODE_VERSION}')
print(f'corpus    {len(CASE_DIRS)} cases')
print(f'tool      {TOOL.name} from this checkout')

prettier  3.9.6, the version .pre-commit-config.yaml pins
node      v26.3.0
corpus    123 cases
tool      unwrap-markdown-prose-py from this checkout


## The measurement

Prettier is invoked through its JavaScript API rather than its command line.
This matters more than it sounds. The `.prettierignore` in this repository
excludes every Markdown file, because prose belongs to this repository's own
hook; a command-line run started anywhere inside the tree therefore formats
nothing, exits zero, prints no diagnostic, and would report near-perfect
agreement by measuring Prettier as the identity function. The API reads no
configuration file and no ignore file, so the option object below is the whole
configuration.

The next cell writes a small Node bridge into a scratch directory rather than
into this repository, runs it once over the whole corpus, and reads back one
JSON document. A case that raises inside Prettier aborts the run: excluding it
would remove it from the numerator and the denominator at once, which quietly
improves the result.

Every guard below ends the run rather than warning, and the figures are
reported after all of them. That the Prettier npm resolved is the one this
repository pins, and that the bridge ran that same build. That the number of
cases measured is the number of cases present. That every case declares itself
unchanged exactly when it ships no expected output. That no input already
contains a Prettier marker, which the reverse translation two sections down
would otherwise corrupt. That no normalizing pass removed a non-blank line.
That the symmetric variant of the normalization is still far from the
asymmetric one. And that this tool still reproduces every expected output byte
for byte.

In [2]:
from dataclasses import dataclass

BRIDGE_SOURCE = r"""import fs from "node:fs";
import path from "node:path";

const [modulePath, casesDir] = process.argv.slice(2);
const prettier = (await import(modulePath)).default;

// Prettier's own defaults, each named, because the claim is about
// `prettier --prose-wrap never` with no configuration; proseWrap is the only
// thing that varies. Formatting inside a fence is off, or a translated marker
// in a Markdown fence would be honored inside that embedded document.
const OPTIONS = {
  parser: "markdown",
  printWidth: 80,
  tabWidth: 2,
  useTabs: false,
  endOfLine: "lf",
  embeddedLanguageFormatting: "off",
  plugins: [],
};
const format = (text, proseWrap) =>
  prettier.format(text, { ...OPTIONS, proseWrap });

const toPrettier = (text) =>
  text
    .replace(/unwrap-ignore-start/g, "prettier-ignore-start")
    .replace(/unwrap-ignore-end/g, "prettier-ignore-end")
    .replace(/unwrap-ignore/g, "prettier-ignore");
const fromPrettier = (text) =>
  text
    .replace(/prettier-ignore-start/g, "unwrap-ignore-start")
    .replace(/prettier-ignore-end/g, "unwrap-ignore-end")
    .replace(/prettier-ignore/g, "unwrap-ignore");

const lines = (text) => text.split("\n").length;
const solid = (text) =>
  text.split("\n").filter((line) => line.trim() !== "").length;

const cases = [];
for (const slug of fs.readdirSync(casesDir).sort()) {
  const dir = path.join(casesDir, slug);
  if (!fs.statSync(dir).isDirectory()) continue;

  const meta = Object.fromEntries(
    fs
      .readFileSync(path.join(dir, "case.txt"), "utf8")
      .split("\n")
      .filter((line) => line.includes(":"))
      .map((line) => [
        line.slice(0, line.indexOf(":")).trim(),
        line.slice(line.indexOf(":") + 1).trim(),
      ]),
  );
  const declaredUnchanged = meta.expected === "unchanged";
  const hasExpected = fs.existsSync(path.join(dir, "expected.md"));
  if (declaredUnchanged === hasExpected) {
    throw new Error(slug + ": case.txt and expected.md disagree");
  }

  // Node reads without newline translation, so a CRLF input arrives as it
  // sits on disk. Python's text mode would rewrite it.
  const input = fs.readFileSync(path.join(dir, "input.md"), "utf8");
  const expected = hasExpected
    ? fs.readFileSync(path.join(dir, "expected.md"), "utf8")
    : input;
  if (/prettier-ignore/.test(input)) {
    throw new Error(slug + ": the input already carries a prettier marker");
  }

  const prettierAnswer = await format(input, "never");
  const prettierSide = await format(prettierAnswer, "preserve");
  const toolSide = await format(expected, "preserve");
  const prettierCanonical = await format(prettierSide, "never");
  const toolCanonical = await format(toolSide, "never");

  const hasMarker = /unwrap-ignore/.test(input);
  let markerActive = false;
  let agreeTranslated = false;
  if (hasMarker) {
    const translated = fromPrettier(await format(toPrettier(input), "never"));
    markerActive = translated !== prettierAnswer;
    agreeTranslated = (await format(translated, "preserve")) === toolSide;
  }

  cases.push({
    slug,
    declared_unchanged: declaredUnchanged,
    has_marker: hasMarker,
    marker_active: markerActive,
    agree: prettierSide === toolSide,
    agree_translated: agreeTranslated,
    byte_identical: prettierAnswer === expected,
    same_document: prettierCanonical === toolCanonical,
    retained_prettier: lines(prettierSide) - lines(prettierCanonical),
    retained_tool: lines(toolSide) - lines(toolCanonical),
    crlf: /\r\n/.test(input),
    crlf_kept_by_tool: /\r\n/.test(expected),
    crlf_kept_by_prettier: /\r\n/.test(prettierAnswer),
    // proseWrap: preserve still pulls a link reference definition split over
    // two lines back onto one, so this counts what a pass removed.
    lines_lost:
      solid(prettierSide) < solid(prettierAnswer) ||
      solid(toolSide) < solid(expected),
    symmetric_agree: (await format(expected, "never")) === prettierAnswer,
    prettier_output: prettierAnswer,
  });
}

process.stdout.write(JSON.stringify({ version: prettier.version, cases }));
"""


# Each family is a line in the README's list of what this tool leaves alone,
# matched on the case slug, so a new case lands in its family without an edit.
FAMILIES = (
    ('badge', 'badge blocks'),
    ('label', 'label rows'),
    ('speaker', 'speaker turns'),
)

# How far apart the two normalizations have to be, in points, before the
# asymmetric one is believed. The failure it guards against is a one-character
# edit that closes the gap, so the threshold sits well below any gap measured.
SYMMETRIC_MARGIN = 30.0


@dataclass(frozen=True, slots=True)
class Case:
    """One corpus case, and what each of the two programs did with it."""

    slug: str
    declared_unchanged: bool
    has_marker: bool
    marker_active: bool
    agree: bool
    agree_translated: bool
    byte_identical: bool
    same_document: bool
    retained_prettier: int
    retained_tool: int
    crlf: bool
    crlf_kept_by_tool: bool
    crlf_kept_by_prettier: bool
    lines_lost: bool
    symmetric_agree: bool
    prettier_output: str

    @property
    def family(self) -> str:
        """Return the name of the group this case's disagreement belongs to."""
        if self.has_marker:
            return 'the ignore directive'
        if not self.same_document:
            return 'a different document'
        for keyword, name in FAMILIES:
            if keyword in self.slug:
                return name
        return 'other structural guards'


def answer_of(case_dir: Path) -> bytes:
    """Return the bytes this tool is expected to produce for one case."""
    expected = case_dir / 'expected.md'
    source = expected if expected.exists() else case_dir / 'input.md'
    return source.read_bytes()


def copy_the_corpus(case_dirs: list[Path], workspace: Path) -> dict[str, Path]:
    """Copy every case input into ``workspace``, keyed by its case slug.

    The tool is run over the copies. It is never run inside ``corpus/``, whose
    cases pin trailing spaces and CRLF that a writing pass would spend.
    """
    shutil.rmtree(workspace, ignore_errors=True)
    workspace.mkdir(parents=True)
    copies = {}
    for case_dir in case_dirs:
        copy = workspace / f'{case_dir.name}.md'
        shutil.copyfile(case_dir / 'input.md', copy)
        copies[case_dir.name] = copy
    return copies


BRIDGE = SCRATCH / 'corpus.mjs'
BRIDGE.write_text(BRIDGE_SOURCE, encoding='utf-8')
payload = json.loads(
    run(
        ['node', str(BRIDGE), str(PRETTIER_MODULE), str(CORPUS)],
        'the bridge did not finish, so the sample is incomplete',
    )
)
CASES = tuple(Case(**row) for row in payload['cases'])

if payload['version'] != PRETTIER_VERSION:
    raise SystemExit(f'the bridge ran prettier {payload["version"]}')
if len(CASES) != len(CASE_DIRS):
    raise SystemExit(f'{len(CASES)} cases measured of {len(CASE_DIRS)} present')
lost = [case.slug for case in CASES if case.lines_lost]
if lost:
    raise SystemExit(f'a normalizing pass removed a line from {lost}')

copies = copy_the_corpus(CASE_DIRS, SCRATCH / 'corpus-copy')
run(
    [str(TOOL), '--write', *(str(path) for path in copies.values())],
    'the tool refused a corpus input',
)
missed = [
    case_dir.name
    for case_dir in CASE_DIRS
    if copies[case_dir.name].read_bytes() != answer_of(case_dir)
]
if missed:
    raise SystemExit(f'the tool no longer reproduces {missed}')

PLAIN = tuple(case for case in CASES if not case.has_marker)
INERT = tuple(case for case in CASES if case.has_marker and not case.marker_active)
ACTIVE = tuple(case for case in CASES if case.has_marker and case.marker_active)
DECLINED = tuple(case for case in CASES if case.declared_unchanged)
REWRITTEN = tuple(case for case in CASES if not case.declared_unchanged)
DISAGREE = tuple(case for case in CASES if not case.agree)

asymmetric = sum(case.agree for case in CASES)
symmetric = sum(case.symmetric_agree for case in CASES)
points = (symmetric - asymmetric) / len(CASES) * 100

print(f'cases measured                      {len(CASES)}')
print(f'of them, declared unchanged         {len(DECLINED)}')
print(f'expected outputs reproduced         {len(CASE_DIRS) - len(missed)}')
print(f'normalizing passes that joined      {len(lost)}')
print()
print(f'agreement, normalized as designed   {asymmetric} of {len(CASES)}')
print(f'agreement, both sides through never {symmetric} of {len(CASES)}')
print(f'the difference between them         {points:.0f} points')
print()
print('The second figure is what this measurement becomes when someone')
print('simplifies it. It is the agreement of prettier with itself, and every')
print('document this tool declined to touch counts toward it.')
if points < SYMMETRIC_MARGIN:
    raise SystemExit(
        f'the symmetric control collapsed to {points:.0f} points: '
        'check the normalization'
    )

cases measured                      123
of them, declared unchanged         57
expected outputs reproduced         123
normalizing passes that joined      0

agreement, normalized as designed   61 of 123
agreement, both sides through never 122 of 123
the difference between them         50 points

The second figure is what this measurement becomes when someone
simplifies it. It is the agreement of prettier with itself, and every
document this tool declined to touch counts toward it.


## Axis 1: how they break prose

The first question is the plain one, asked of the cases that carry no ignore
marker at all: given the same document, do the two programs put the line breaks
in the same places?

This is the number most people mean by parity, and it is also the number least
able to carry the weight put on it. The cases here are not ordinary documents.
Most of them exist because someone decided a specific shape had to survive
contact with a formatter, and a good many of them are cases where the right
answer is to change nothing. The last section says what that does to the
percentage.

In [3]:
from IPython.display import Markdown, display


def rate(agreeing: int, total: int) -> str:
    """Return a count against a total, with the percentage after it."""
    return f'{agreeing} of {total} ({agreeing / total * 100:.0f}%)'


plain_declined = [case for case in PLAIN if case.declared_unchanged]
plain_rewritten = [case for case in PLAIN if not case.declared_unchanged]

table = [
    '| cases carrying no ignore marker | how many | prettier reaches the same answer |',
    '| -- | --: | --: |',
]
append_to_table = table.append
for label, group in (
    ('this tool rewrites the document', plain_rewritten),
    ('this tool leaves the document alone', plain_declined),
    ('either way', list(PLAIN)),
):
    agreeing = sum(case.agree for case in group)
    append_to_table(f'| {label} | {len(group)} | {rate(agreeing, len(group))} |')
display(Markdown(chr(10).join(table)))

print('The two rows differ because they are two different questions. Agreeing')
print('about how to rewrite a paragraph is the easier one: both programs join')
print('prose, so they agree whenever nothing else is in the way. Agreeing to')
print('leave a document alone is the harder one, and it is where this tool')
print('spends most of its code.')

| cases carrying no ignore marker | how many | prettier reaches the same answer |
| -- | --: | --: |
| this tool rewrites the document | 29 | 19 of 29 (66%) |
| this tool leaves the document alone | 31 | 20 of 31 (65%) |
| either way | 60 | 39 of 60 (65%) |

The two rows differ because they are two different questions. Agreeing
about how to rewrite a paragraph is the easier one: both programs join
prose, so they agree whenever nothing else is in the way. Agreeing to
leave a document alone is the harder one, and it is where this tool
spends most of its code.


## Axis 2: the ignore directive

Both programs let a comment exempt a paragraph, and both spell it almost the
same way. This tool takes `<!-- unwrap-ignore -->` and a start and end marker
around a run of paragraphs; Prettier takes `<!-- prettier-ignore -->` and its
own start and end pair. The names were chosen to follow Prettier's, which the
README says outright.

Comparing them as written therefore measures the spelling rather than the
feature: Prettier does not recognize this tool's marker, so every exemption in
the corpus simply fails to happen. The fair test substitutes the name, runs
Prettier, and substitutes it back. That substitution is an analytical device
and nothing more. This tool does not perform it, does not recognize Prettier's
spelling, and gains nothing from the translated number except an honest answer
to the question of whether the two features behave the same way.

The cases split three ways rather than two. Some carry a marker Prettier
ignores even after translation: the translated input gets back the same output
the untranslated one did. A marker inside a fence, inside front matter or inside
an HTML block is one such case and not the only one, so the group is defined by
that measurement rather than by where the marker sits, and those cases cannot
move. The rest carry a marker Prettier acts on once it is spelled its way, and
those are the ones the feature comparison is about. The cell below reports all three groups,
as written and translated, and names every case whose verdict the translation
changes in either direction.

In [4]:
marker_cases = [case for case in CASES if case.has_marker]
gained = [case for case in marker_cases if not case.agree and case.agree_translated]
lost_by_translation = [
    case for case in marker_cases if case.agree and not case.agree_translated
]

table = [
    '| marker cases | how many | agree as written | agree translated |',
    '| -- | --: | --: | --: |',
]
append_to_table = table.append
for label, group in (
    ('prettier ignores the marker either way', INERT),
    ('prettier acts on the marker once translated', ACTIVE),
):
    append_to_table(
        f'| {label} | {len(group)} '
        f'| {sum(case.agree for case in group)} '
        f'| {sum(case.agree_translated for case in group)} |'
    )
append_to_table('')
append_to_table(
    f'The second row is the feature comparison. The first cannot move: for '
    f'those {len(INERT)} cases, prettier returns the same output whether the '
    'marker is translated or not, so renaming it changes nothing. A marker '
    'inside a fence, inside front matter or inside an HTML block is one way to '
    'get there, not the only one. Counting all marker cases together dilutes '
    'the result and hides which direction each case moved in.'
)
display(Markdown(chr(10).join(table)))

print(f'cases the substitution decides the other way: {len(gained)} gained,')
print(f'{len(lost_by_translation)} lost.')
for case in lost_by_translation:
    print(f'     {case.slug}')

| marker cases | how many | agree as written | agree translated |
| -- | --: | --: | --: |
| prettier ignores the marker either way | 37 | 20 | 20 |
| prettier acts on the marker once translated | 26 | 2 | 24 |

The second row is the feature comparison. The first cannot move: for those 37 cases, prettier returns the same output whether the marker is translated or not, so renaming it changes nothing. A marker inside a fence, inside front matter or inside an HTML block is one way to get there, not the only one. Counting all marker cases together dilutes the result and hides which direction each case moved in.

cases the substitution decides the other way: 22 gained,
0 lost.


## Axis 3: line endings

A file written on Windows uses CRLF, and this tool returns it with CRLF. That
is one of the two guarantees this tool makes about what it will not do, and it
is the guarantee the joining measurement cannot see.

It cannot see it because the normalization is applied to both sides. Whatever
Prettier does to line endings on its own answer it also does to this tool's
answer, so the two cancel and the difference disappears. Setting `endOfLine` to
something other than the default does not rescue it; the cancellation is
structural, not a consequence of the value chosen. So this axis is measured on
raw bytes, before any normalization, and reported on its own.

In [5]:
crlf_cases = [case for case in CASES if case.crlf]
kept_by_tool = sum(case.crlf_kept_by_tool for case in crlf_cases)
kept_by_prettier = sum(case.crlf_kept_by_prettier for case in crlf_cases)
hidden = [case for case in crlf_cases if case.agree]

print(f'corpus cases whose input uses CRLF     {len(crlf_cases)}')
print(f'this tool returns CRLF for             {kept_by_tool} of {len(crlf_cases)}')
print(f'prettier returns CRLF for              {kept_by_prettier} of {len(crlf_cases)}')
print(f'scored as agreeing in the figures above {len(hidden)} of {len(crlf_cases)}')
print()
print('Measured on the bytes each program produced, before any normalization.')
print('The last row is the point. The normalization applies the same')
print('line-ending rewrite to both answers, so it cancels, and a case where')
print('one program preserved the file and the other did not can still be')
print('counted as agreement about where the line breaks go. That is the right')
print('answer to the joining question and the wrong answer to the question a')
print('reader on a CRLF repository is actually asking.')

corpus cases whose input uses CRLF     4
this tool returns CRLF for             4 of 4
prettier returns CRLF for              0 of 4
scored as agreeing in the figures above 1 of 4

Measured on the bytes each program produced, before any normalization.
The last row is the point. The normalization applies the same
line-ending rewrite to both answers, so it cancels, and a case where
one program preserved the file and the other did not can still be
counted as agreement about where the line breaks go. That is the right
answer to the joining question and the wrong answer to the question a
reader on a CRLF repository is actually asking.


## Where they differ

Every disagreement is worth a name. The cell below sorts them into families and
prints how many cases each family holds, then shows one case from each in full.

Which case stands for a family is drawn rather than chosen. The draw is a pure
function of the family, a nudge, and the slug, so the same numbers always
produce the same cases, and a family's pick moves when the corpus grows only if
a new case happens to draw lower. Bumping a family's nudge redraws that family
and nothing else.

That is worth the machinery because the alternative failed here. This section
used to name three cases by hand, and the three had something in common that
nobody chose deliberately: Prettier came off worse in all of them. A list
written once cannot tell you how many candidates were passed over, and a page
comparing two programs is exactly where that matters. A nudge that changed is
visible in the diff.

The price is that the corpus is adversarial, so a draw can land on a case that
reads badly — a vertical tab, a digit that is not an ASCII digit. The nudge is
there for that, and using it is honest as long as it shows.

Most of the families are a line in the README's list of what this tool leaves
alone, and each exists because joining across that shape cost something. A
badge block is a run of link-only lines that a Markdown file renders as one row
of images either way, so what the fold costs is the diff rather than the page:
one badge to a line makes adding a badge a one-line change. A label row is a
bold key with its value, on a line of its own, which joining turns into one
run-on line — and in a pull request body or an issue comment, where a soft
break renders as a line break, that reaches the reader as well. A speaker turn
is a name, a colon, and the line that follows it. An alert marker is GitHub's
own syntax, which puts the marker alone on its line, so a formatter that pulls
the prose up beside it has written something GitHub stops rendering as an
alert.

Two families are not about a shape this tool protects. The ignore directive is
not a disagreement about prose at all: Prettier has never heard of this tool's
marker, reads it as an ordinary comment, and joins the paragraph the marker was
there to protect.

The last family holds documents the two programs did not read the same way, and
it is the one place on this page where this tool is the one that rewrites and
Prettier is the one that declines. A page that printed only the differences
flattering to the tool it ships with would be an advertisement.

The fragments below show this tool's answer exactly as the corpus holds it, not
the normalized form the comparison uses. The normalized form is right for
counting and wrong for reading: it re-indents continuation lines and adds
blockquote markers that neither program put there.

In [6]:
import hashlib
from collections import Counter

# Bump a family's nudge to draw a different case for it and nothing else.
# Nothing here reads this at run time except the draw, so a nudge that changed
# is a line in the diff rather than a silent re-pick.
NUDGE: dict[str, int] = {}


def drawn(family: str, slugs: list[str]) -> str:
    """Return the case drawn to stand for one family of disagreement."""
    nudge = NUDGE.get(family, 0)
    return min(
        slugs,
        key=lambda slug: hashlib.sha256(
            f'{family}|{nudge}|{slug}'.encode()
        ).hexdigest(),
    )


# Retained breaks, rather than raw line counts: a blank line prettier inserts
# after an HTML block, or a blockquote marker it adds, moves a line count with
# no joining decision made. Subtracting a full join from each side leaves only
# the breaks that side chose to keep.
same = [case for case in DISAGREE if case.same_document]
joined_more = [case for case in same if case.retained_prettier < case.retained_tool]
joined_less = [case for case in same if case.retained_prettier > case.retained_tool]
structural = [case for case in DISAGREE if not case.same_document]

counted = Counter(case.family for case in DISAGREE)
table = ['| family | disagreements |', '| -- | --: |']
append_to_table = table.append
for name, count in counted.most_common():
    append_to_table(f'| {name} | {count} |')
append_to_table(f'| **all of them** | **{len(DISAGREE)}** |')
display(Markdown(chr(10).join(table)))

print(f'disagreements                             {len(DISAGREE)}')
print(f'  the same document, broken differently   {len(same)}')
print(f'    prettier joined more                  {len(joined_more)}')
print(f'    prettier joined less                  {len(joined_less)}')
print(f'  a structurally different document       {len(structural)}')
print()
print('The last row is a document the two programs did not read the same way,')
print('so neither of them declined to join anything in it. It is separated out')
print('because counting it as a joining decision would be wrong in kind rather')
print('than wrong by one.')

by_slug = {case.slug: case for case in CASES}
by_family: dict[str, list[str]] = {}
for case in DISAGREE:
    by_family.setdefault(case.family, []).append(case.slug)

# One case per family, every family, largest family first. Drawing from the
# families rather than naming cases is what makes those three properties hold
# by construction: there is nothing left here for an assertion to catch.
FRAGMENTS = tuple(
    drawn(family, sorted(by_family[family])) for family, _ in counted.most_common()
)

document = []
for slug in FRAGMENTS:
    case = by_slug[slug]
    # Bytes, like answer_of: text mode would translate a CRLF case's endings.
    source = (CORPUS / slug / 'input.md').read_bytes().decode('utf-8')
    answer = answer_of(CORPUS / slug).decode('utf-8')
    document += [
        f'**{slug.replace("-", " ")}** — {case.family}',
        '',
        'the document as it arrives:',
        '',
        f'```markdown\n{source}```',
        '',
    ]
    for name, text in (('this tool', answer), ('prettier', case.prettier_output)):
        if text == source:
            document += [f'{name} returns it exactly as it arrived.', '']
        else:
            document += [f'{name}:', '', f'```markdown\n{text}```', '']
display(Markdown(chr(10).join(document)))

| family | disagreements |
| -- | --: |
| the ignore directive | 41 |
| other structural guards | 9 |
| badge blocks | 6 |
| label rows | 3 |
| speaker turns | 2 |
| a different document | 1 |
| **all of them** | **62** |

disagreements                             62
  the same document, broken differently   61
    prettier joined more                  61
    prettier joined less                  0
  a structurally different document       1

The last row is a document the two programs did not read the same way,
so neither of them declined to join anything in it. It is separated out
because counting it as a joining decision would be wrong in kind rather
than wrong by one.


**a quoted multiline directive exempts the quoted paragraph** — the ignore directive

the document as it arrives:

```markdown
> <!--
> unwrap-ignore
> -->
> Left alone
> on purpose.
```

this tool returns it exactly as it arrived.

prettier:

```markdown
> <!--
> unwrap-ignore
> -->
>
> Left alone on purpose.
```

**bare pipe stays wrapped** — other structural guards

the document as it arrives:

```markdown
Alpha beta with a plain | pipe that yields
null on no match.
```

this tool returns it exactly as it arrived.

prettier:

```markdown
Alpha beta with a plain | pipe that yields null on no match.
```

**a deeper quoted badge line stays in the run** — badge blocks

the document as it arrives:

```markdown
> > [![Alpha](https://img.example.com/a.svg)][ref]
> [![Bravo](https://img.example.com/b.svg)][ref]
Prose that
wraps.
```

this tool:

```markdown
> > [![Alpha](https://img.example.com/a.svg)][ref]
> [![Bravo](https://img.example.com/b.svg)][ref]
Prose that wraps.
```

prettier:

```markdown
> > [![Alpha](https://img.example.com/a.svg)][ref] [![Bravo](https://img.example.com/b.svg)][ref] Prose that wraps.
```

**bold label rows are preserved** — label rows

the document as it arrives:

```markdown
**Detect:** reads the title
**Action:** blocks the merge
```

this tool returns it exactly as it arrived.

prettier:

```markdown
**Detect:** reads the title **Action:** blocks the merge
```

**inline speaker turns keep their boundaries** — speaker turns

the document as it arrives:

```markdown
Alex: First turn wraps
over two lines.
Sam: Second turn wraps
over two lines.
```

this tool:

```markdown
Alex: First turn wraps over two lines.
Sam: Second turn wraps over two lines.
```

prettier:

```markdown
Alex: First turn wraps over two lines. Sam: Second turn wraps over two lines.
```

**a kelvin sign folds into an html closing tag** — a different document

the document as it arrives:

```markdown
<blockquote>
raw html here
</BLOCKQUOTE>
wrapped prose
here.
```

this tool:

```markdown
<blockquote>
raw html here
</BLOCKQUOTE>
wrapped prose here.
```

prettier returns it exactly as it arrived.


## Two mechanisms, on documents this tool was not written for

The corpus says what happens to the documents someone thought to write down.
This section asks what happens to two documents nobody did, and it generates
them here rather than reading anything out of this repository, so that adding a
file to the tree cannot change a figure on this page.

The first is a badge block: five shield links, each on its own line, the shape
at the top of a great many README files. The second is a fenced code block
tagged as Markdown, holding a wrapped paragraph.

The second one demonstrates a mechanism the measurement above deliberately
switches off. Prettier formats the contents of a fenced block in the language
the fence is tagged with, and Markdown is one of the languages it knows, so a
Markdown fence is reformatted as a document in its own right. The measurement
pins `embeddedLanguageFormatting` to `off` to keep the comparison about
joining, which is a choice that flatters Prettier: a reader who runs Prettier
with no configuration gets the default of `auto` and gets this as well. The
cell below runs both settings over fences with several tags, so the boundary is
visible rather than described.

In [7]:
FIXTURES_BRIDGE_SOURCE = r"""import fs from "node:fs";
import path from "node:path";

const [modulePath, fixturesDir] = process.argv.slice(2);
const prettier = (await import(modulePath)).default;

const OPTIONS = {
  parser: "markdown",
  proseWrap: "never",
  printWidth: 80,
  tabWidth: 2,
  useTabs: false,
  endOfLine: "lf",
  plugins: [],
};

const fixtures = [];
for (const name of fs.readdirSync(fixturesDir).sort()) {
  const text = fs.readFileSync(path.join(fixturesDir, name), "utf8");
  fixtures.push({
    name,
    embedded_off: await prettier.format(text, {
      ...OPTIONS,
      embeddedLanguageFormatting: "off",
    }),
    embedded_auto: await prettier.format(text, {
      ...OPTIONS,
      embeddedLanguageFormatting: "auto",
    }),
  });
}

process.stdout.write(JSON.stringify(fixtures));
"""


# Five shield links, one per line, the shape at the top of many README files.
BADGES = (
    ('build', 'build-passing-brightgreen'),
    ('coverage', 'coverage-99-brightgreen'),
    ('license', 'license-MIT-blue'),
    ('version', 'version-1.2.3-orange'),
    ('chat', 'chat-welcome-blueviolet'),
)

# One wrapped paragraph inside a fence under each tag. Prettier formats a
# fenced block as the language its tag names, and `markdown` and `md` name
# Markdown.
FENCE_TAGS = 'markdown', 'md', 'text', 'js', 'totally-unknown', ''
UNTAGGED = 'untagged'
FENCED_PROSE = 'Prose inside the fence\nwrapped over two lines.\n'


def write_fixtures(folder: Path) -> dict[str, str]:
    """Generate the fixture documents in ``folder`` and return them by filename."""
    shutil.rmtree(folder, ignore_errors=True)
    folder.mkdir(parents=True)
    written = {
        'badge-block.md': ''.join(
            f'[![{label}](https://img.shields.io/badge/{slug}.svg)]'
            f'(https://example.com/{label})\n'
            for label, slug in BADGES
        )
    }
    for tag in FENCE_TAGS:
        name = f'fence-{tag or UNTAGGED}.md'
        written[name] = f'```{tag}\n{FENCED_PROSE}```\n'
    for name, text in written.items():
        (folder / name).write_text(text, encoding='utf-8')
    return written


FIXTURES = SCRATCH / 'fixtures'
source = write_fixtures(FIXTURES)

# The tool is run over a second copy, because it rewrites in place and the
# comparison needs the documents as generated.
worked = SCRATCH / 'fixtures-tool'
shutil.rmtree(worked, ignore_errors=True)
shutil.copytree(FIXTURES, worked)
run(
    [str(TOOL), '--write', *(str(path) for path in sorted(worked.iterdir()))],
    'the tool refused a generated fixture',
)

FIXTURE_BRIDGE = SCRATCH / 'fixtures.mjs'
FIXTURE_BRIDGE.write_text(FIXTURES_BRIDGE_SOURCE, encoding='utf-8')
formatted = {
    row['name']: row
    for row in json.loads(
        run(
            ['node', str(FIXTURE_BRIDGE), str(PRETTIER_MODULE), str(FIXTURES)],
            'the fixture bridge did not finish',
        )
    )
}


def verdict(before: str, after: str) -> str:
    """Return whether a program changed a document, written for a table cell."""
    return 'left alone' if before == after else '**joined**'


badge = 'badge-block.md'
print('a badge block of five shield links, as generated:')
print()
print(source[badge], end='')
print()
print('prettier:')
print()
print(formatted[badge]['embedded_off'], end='')
print()
badge_after = (worked / badge).read_text(encoding='utf-8')
print(f'this tool: {verdict(source[badge], badge_after)}')
print()

table = [
    '| fence tag | this tool | prettier, embedded off | prettier, embedded auto |',
    '| -- | -- | -- | -- |',
]
append_to_table = table.append
for tag in FENCE_TAGS:
    name = f'fence-{tag or UNTAGGED}.md'
    label = f'`{tag}`' if tag else 'untagged'
    before = source[name]
    after = (worked / name).read_text(encoding='utf-8')
    append_to_table(
        f'| {label} '
        f'| {verdict(before, after)} '
        f'| {verdict(before, formatted[name]["embedded_off"])} '
        f'| {verdict(before, formatted[name]["embedded_auto"])} |'
    )
display(Markdown(chr(10).join(table)))

a badge block of five shield links, as generated:

[![build](https://img.shields.io/badge/build-passing-brightgreen.svg)](https://example.com/build)
[![coverage](https://img.shields.io/badge/coverage-99-brightgreen.svg)](https://example.com/coverage)
[![license](https://img.shields.io/badge/license-MIT-blue.svg)](https://example.com/license)
[![version](https://img.shields.io/badge/version-1.2.3-orange.svg)](https://example.com/version)
[![chat](https://img.shields.io/badge/chat-welcome-blueviolet.svg)](https://example.com/chat)

prettier:

[![build](https://img.shields.io/badge/build-passing-brightgreen.svg)](https://example.com/build) [![coverage](https://img.shields.io/badge/coverage-99-brightgreen.svg)](https://example.com/coverage) [![license](https://img.shields.io/badge/license-MIT-blue.svg)](https://example.com/license) [![version](https://img.shields.io/badge/version-1.2.3-orange.svg)](https://example.com/version) [![chat](https://img.shields.io/badge/chat-welcome-blueviolet.s

| fence tag | this tool | prettier, embedded off | prettier, embedded auto |
| -- | -- | -- | -- |
| `markdown` | left alone | left alone | **joined** |
| `md` | left alone | left alone | **joined** |
| `text` | left alone | left alone | left alone |
| `js` | left alone | left alone | left alone |
| `totally-unknown` | left alone | left alone | left alone |
| untagged | left alone | left alone | left alone |

## The three axes on one chart

The chart is written next to this notebook rather than embedded in it, and as
SVG rather than PNG. This repository routes PNG files through Git LFS, so a PNG
chart would be committed as a pointer and appear as nothing at all to anyone
cloning without git-lfs; and an embedded PNG is stored as base64 text, which
the two spell-checking hooks read as prose.

The left panel is agreement, as a share of each group, with the counts written
on the bars. The right panel is the corpus split into the documents this tool
rewrites and the documents it leaves alone, showing what Prettier does with
each half.

In [8]:
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.axes import Axes
from matplotlib.figure import Figure

# Element ids are derived from this value rather than from memory addresses, so
# two runs over the same numbers produce the same file.
mpl.rcParams['svg.hashsalt'] = 'markdown-prose-hooks'

# The second is a neutral, because it is the remainder of a whole rather than
# a series competing with the first for attention.
FILLED, REMAINDER = '#1f77b4', '#bbbbbb'


def hide_spines(*all_axes: Axes) -> None:
    """Remove the top and right borders from each chart."""
    for axes in all_axes:
        axes.spines['top'].set_visible(False)
        axes.spines['right'].set_visible(False)


def save_chart(figure: Figure, name: str) -> None:
    """Write ``figure`` to the docs directory as the bytes that get committed.

    matplotlib writes the current date into the SVG metadata and leaves
    trailing spaces that this repository's own hook removes on commit. Both
    are removed here, so the committed chart is the file this run wrote and a
    run that measured nothing new leaves no change behind.
    """
    path = REPO / 'docs' / name
    figure.savefig(path, metadata={'Date': None})
    plt.close(figure)
    text = path.read_text(encoding='utf-8')
    path.write_text(
        chr(10).join(line.rstrip() for line in text.split(chr(10))),
        encoding='utf-8',
    )
    print(f'chart written to docs/{name}')


def shares(
    axes: Axes,
    rows: list[tuple[str, int, int]],
    legend: list[str],
    height: float,
) -> None:
    """Draw one full-width bar per row, split at the share the count covers.

    Every count is written to the right of its own bar rather than inside it,
    because a row where the count is zero has no bar to write inside.
    """
    labels = [label for label, _, _ in rows]
    filled = [part / total * 100 for _, part, total in rows]
    axes.barh(labels, filled, height=height, color=FILLED, label=legend[0])
    axes.barh(
        labels,
        [100 - value for value in filled],
        left=filled,
        height=height,
        color=REMAINDER,
        label=legend[1],
    )
    for index, row in enumerate(rows):
        axes.text(104, index, f'{row[1]} of {row[2]}', va='center', fontsize=9)
    axes.set_xlim(0, 130)
    axes.set_xticks([0, 50, 100])
    axes.set_xlabel('share of the group (%)')
    axes.invert_yaxis()
    axes.grid(alpha=0.3, axis='x')
    axes.legend(
        loc='upper center',
        bbox_to_anchor=(0.5, -0.16),
        ncol=2,
        frameon=False,
        fontsize=9,
    )


agreement_rows = [
    ('prose joining, no marker', sum(case.agree for case in PLAIN), len(PLAIN)),
    ('marker inert to prettier', sum(case.agree for case in INERT), len(INERT)),
    ('marker active, as written', sum(case.agree for case in ACTIVE), len(ACTIVE)),
    (
        'marker active, translated',
        sum(case.agree_translated for case in ACTIVE),
        len(ACTIVE),
    ),
    (
        'documents this tool rewrites',
        sum(case.agree for case in REWRITTEN),
        len(REWRITTEN),
    ),
    (
        'documents it leaves alone',
        sum(case.agree for case in DECLINED),
        len(DECLINED),
    ),
]
ending_rows = [
    ('this tool', kept_by_tool, len(crlf_cases)),
    ('prettier', kept_by_prettier, len(crlf_cases)),
]

figure, (left, right) = plt.subplots(
    1, 2, figsize=(11, 4.2), gridspec_kw={'width_ratios': [2, 1]}
)
shares(left, agreement_rows, ['the two agree', 'they differ'], 0.62)
left.set_title('Where the two programs agree')
# The last two rows count the whole corpus split another way, rather than
# another bucket of the split above them.
left.axhline(3.5, color='#444444', linewidth=0.8)
shares(right, ending_rows, ['CRLF preserved', 'rewritten to LF'], 0.4)
right.set_title('Files that arrive with CRLF')
hide_spines(left, right)
figure.tight_layout()
save_chart(figure, 'prettier-parity.svg')

chart written to docs/prettier-parity.svg


![Agreement by group, and what Prettier does with each half of the corpus](prettier-parity.svg)

## What this does and does not say

**The corpus is this tool's own specification.** Every case in it exists
because somebody decided that behavior mattered enough to pin, and a large
share of them exist to pin the decision to leave a document alone. Several are
deliberately hostile: a Kelvin sign that looks like a capital K, a digit that is
not an ASCII digit, a vertical tab, an unterminated run of backticks. Nothing
about that mixture resembles the Markdown in an ordinary repository.

So none of the percentages above is a sample of ordinary Markdown, and none of
them should ever be presented as one. They are agreement on an adversarial
conformance suite, which is a different thing and a harder test. A reader who
takes the first figure as the chance that Prettier will do the same thing to
their README has been misled, and this page would be the thing that misled
them.

The statistic that survives the objection is about a population rather than a
rate, and the final cell prints it: of the documents this tool declines to
touch, how many Prettier rewrites. That one is defensible because the
population is named. It is the whole risk of substituting one program for the
other, stated in the only terms that do not depend on the corpus being
representative of anything.

This page recommends nothing. Which formatter to run is a decision about what a
repository wants left alone, and it is not a decision any number here settles.

In [9]:
import textwrap

declined_rewritten = len(DECLINED) - sum(case.agree for case in DECLINED)

document = [
    '### What was measured',
    '',
    f'- **How they break prose.** Of the {len(PLAIN)} cases carrying no ignore '
    f'marker, the two programs put the line breaks in the same places for '
    f'{sum(case.agree for case in PLAIN)}.',
    f'- **The ignore directive.** Of the {len(ACTIVE)} cases where prettier acts '
    f'on the marker once it is spelled its way, the two agree on '
    f'{sum(case.agree for case in ACTIVE)} as written and '
    f'{sum(case.agree_translated for case in ACTIVE)} translated. The other '
    f'{len(INERT)} marker cases get the same output from prettier whether '
    'the marker is translated or not.',
    f'- **Line endings.** Of the {len(crlf_cases)} cases that arrive with CRLF, '
    f'this tool returns {kept_by_tool} with CRLF and prettier returns '
    f'{kept_by_prettier}.',
    f'- **Which way the disagreements run.** Of {len(DISAGREE)} disagreements, '
    f'prettier joined more in {len(joined_more)} and joined less in '
    f'{len(joined_less)}. The remaining {len(structural)} is a document the two '
    f'programs did not read the same way.',
    f'- **As a drop-in replacement.** Prettier produced the same bytes this tool '
    f'did for {sum(case.byte_identical for case in CASES)} of {len(CASES)} cases.',
    '',
    '### The statistic that survives the objection below',
    '',
    textwrap.fill(
        f'Of the {len(DECLINED)} documents this tool declines to touch, prettier '
        f'rewrites {declined_rewritten}. That is the whole risk of substituting '
        'one program for the other, and it is stated about a named population '
        'rather than as a rate over a corpus that resembles nobody in '
        'particular.',
        width=86,
    ),
    '',
    '### What was not measured',
    '',
    textwrap.fill(
        'What either program does to ordinary Markdown. Every case here exists '
        'because somebody decided that behavior mattered enough to pin, which '
        'is the opposite of a sample. How long either program takes to run, '
        'which is the notebook beside this one. And every difference this page '
        'normalizes away on purpose: prettier also renumbers ordered lists, '
        'rewrites fences, pads table cells and converts emphasis markers, and a '
        'reader who switched would see all of it.',
        width=86,
    ),
]
display(Markdown(chr(10).join(document)))

### What was measured

- **How they break prose.** Of the 60 cases carrying no ignore marker, the two programs put the line breaks in the same places for 39.
- **The ignore directive.** Of the 26 cases where prettier acts on the marker once it is spelled its way, the two agree on 2 as written and 24 translated. The other 37 marker cases get the same output from prettier whether the marker is translated or not.
- **Line endings.** Of the 4 cases that arrive with CRLF, this tool returns 4 with CRLF and prettier returns 0.
- **Which way the disagreements run.** Of 62 disagreements, prettier joined more in 61 and joined less in 0. The remaining 1 is a document the two programs did not read the same way.
- **As a drop-in replacement.** Prettier produced the same bytes this tool did for 39 of 123 cases.

### The statistic that survives the objection below

Of the 57 documents this tool declines to touch, prettier rewrites 37. That is the
whole risk of substituting one program for the other, and it is stated about a named
population rather than as a rate over a corpus that resembles nobody in particular.

### What was not measured

What either program does to ordinary Markdown. Every case here exists because somebody
decided that behavior mattered enough to pin, which is the opposite of a sample. How
long either program takes to run, which is the notebook beside this one. And every
difference this page normalizes away on purpose: prettier also renumbers ordered
lists, rewrites fences, pads table cells and converts emphasis markers, and a reader
who switched would see all of it.

## The artifact

The final cell writes what was measured to `prettier-parity.json`, beside this
notebook. Nothing on this page reads it; it exists so that a check can compare
a later run against this one without re-reading the page, and so that the
Prettier version these figures belong to travels with them.

The names in it are a contract rather than a convenience. The check recomputes
this whole measurement at the pinned Prettier and compares the two documents
number by number, at the same path in each, so a count that gets renamed here
is renamed there in the same change or the check reports it as a figure nothing
computed.

It is written through Prettier itself. This repository's hooks reformat every
JSON file on commit, and Prettier's JSON style is not Python's: it pulls any
array short enough back onto one line. A file written with a plain JSON dumper
would be rewritten the first time it was committed, and the bytes on this page
would not be the bytes in the tree.

In [10]:
JSON_FORMATTER_SOURCE = r"""import fs from "node:fs";

const [modulePath, jsonPath] = process.argv.slice(2);
const prettier = (await import(modulePath)).default;

process.stdout.write(
  await prettier.format(fs.readFileSync(jsonPath, "utf8"), {
    parser: "json",
    printWidth: 80,
    tabWidth: 2,
    useTabs: false,
    endOfLine: "lf",
  }),
);
"""


# Prettier rewrites every JSON file here on commit and pulls any short array
# onto one line, so the artifact is written through prettier to make the
# committed bytes the bytes this run produced.
JSON_FORMATTER = SCRATCH / 'format-json.mjs'
JSON_FORMATTER.write_text(JSON_FORMATTER_SOURCE, encoding='utf-8')

artifact = {
    'prettier_version': PRETTIER_VERSION,
    'corpus_cases': len(CASES),
    'cases_declared_unchanged': len(DECLINED),
    'axis_1_prose_joining': {
        'cases': len(PLAIN),
        'agree': sum(case.agree for case in PLAIN),
    },
    'axis_2_ignore_directives': {
        'cases': len(INERT) + len(ACTIVE),
        'marker_inert': {
            'cases': len(INERT),
            'agree': sum(case.agree for case in INERT),
        },
        'marker_active_as_written': {
            'cases': len(ACTIVE),
            'agree': sum(case.agree for case in ACTIVE),
        },
        'marker_active_translated': {
            'cases': len(ACTIVE),
            'agree': sum(case.agree_translated for case in ACTIVE),
        },
    },
    'axis_3_line_endings': {
        'cases': len(crlf_cases),
        'tool_preserves_crlf': kept_by_tool,
        'prettier_preserves_crlf': kept_by_prettier,
    },
    'overall': {
        'agree': asymmetric,
        'disagree': len(DISAGREE),
        'same_document_broken_differently': len(same),
        'prettier_joined_more': len(joined_more),
        'prettier_joined_less': len(joined_less),
        'structurally_different': len(structural),
        'raw_byte_identical': sum(case.byte_identical for case in CASES),
        'declined_by_tool': len(DECLINED),
        'declined_but_prettier_rewrites': declined_rewritten,
    },
    # The outcome of the draw, not its inputs. The check holds each name to
    # the one property it computes on its own -- that the case is still a
    # disagreement -- which is what catches an artifact edited by hand. The
    # nudges stay in the cell above: recording an input nothing checks is the
    # kind of figure that goes stale without anything going red.
    'disagreement_examples': list(FRAGMENTS),
}

ARTIFACT_PATH = REPO / 'docs' / 'prettier-parity.json'
raw = SCRATCH / 'prettier-parity.json'
raw.write_text(json.dumps(artifact, indent=2), encoding='utf-8')
ARTIFACT_PATH.write_text(
    run(
        ['node', str(JSON_FORMATTER), str(PRETTIER_MODULE), str(raw)],
        'prettier could not format the artifact',
    ),
    encoding='utf-8',
)
print(f'written to docs/{ARTIFACT_PATH.name}, {ARTIFACT_PATH.stat().st_size} bytes')
print()
print(ARTIFACT_PATH.read_text(encoding='utf-8'), end='')

written to docs/prettier-parity.json, 1157 bytes

{
  "prettier_version": "3.9.6",
  "corpus_cases": 123,
  "cases_declared_unchanged": 57,
  "axis_1_prose_joining": {
    "cases": 60,
    "agree": 39
  },
  "axis_2_ignore_directives": {
    "cases": 63,
    "marker_inert": {
      "cases": 37,
      "agree": 20
    },
    "marker_active_as_written": {
      "cases": 26,
      "agree": 2
    },
    "marker_active_translated": {
      "cases": 26,
      "agree": 24
    }
  },
  "axis_3_line_endings": {
    "cases": 4,
    "tool_preserves_crlf": 4,
    "prettier_preserves_crlf": 0
  },
  "overall": {
    "agree": 61,
    "disagree": 62,
    "same_document_broken_differently": 61,
    "prettier_joined_more": 61,
    "prettier_joined_less": 0,
    "structurally_different": 1,
    "raw_byte_identical": 39,
    "declined_by_tool": 57,
    "declined_but_prettier_rewrites": 37
  },
  "disagreement_examples": [
    "a-quoted-multiline-directive-exempts-the-quoted-paragraph",
    "bare-pipe-stay